In [20]:
!pip install spiceypy
!pip install plotly

In [21]:
import datetime
import glob

from matplotlib import pyplot as pyplot
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import spiceypy

In [22]:
# First we need to download some kernels

# Leapseconds
!curl https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls --create-dirs -o kernels/lsk/naif0012.tls

# SPK
!curl https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de432s.bsp --create-dirs -o kernels/spk/de432s.bsp

# PCK
!curl https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de440.tpc --create-dirs -o kernels/pck/gm_de440.tpc

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  5257  100  5257    0     0   6021      0 --:--:-- --:--:-- --:--:--  6084
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:02 --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:03 --:--:--     0
  0     0    0     0    0     0      0      0 --:-

In [23]:
kernel_filepaths = glob.glob("kernels/**/*")
print(kernel_filepaths)

['kernels\\lsk\\naif0012.tls', 'kernels\\pck\\gm_de440.tpc', 'kernels\\spk\\de432s.bsp']


In [24]:
spiceypy.furnsh(kernel_filepaths)

In [25]:
datetime_now = datetime.datetime.today()
datetime_now = datetime_now.strftime("%Y-%m-%dT%H:%M:%S")
print(datetime_now)

2026-09-11T15:31:33


In [26]:
et_now = spiceypy.utc2et(datetime_now)
print(et_now)

842412762.1824818


In [27]:
earth_state_wrt_sun, earth_sun_light_time = spiceypy.spkgeo(targ = 399,
                                                        et = et_now,
                                                        ref="ECLIPJ2000",
                                                        obs=10
                                                      )
print(earth_state_wrt_sun)
print(earth_sun_light_time)

[ 1.47603963e+08 -2.98924690e+07  1.09216049e+03  5.43144188e+00
  2.90968226e+01 -7.19173269e-04]
502.3489611411556


In [28]:
earth_sun_lt_min = earth_sun_light_time / 60
print(f"Light travel time between Earth and Sun: {earth_sun_lt_min}")

Light travel time between Earth and Sun: 8.372482685685927


In [29]:
earth_sun_dist_km = np.linalg.norm(earth_state_wrt_sun[:3])
print(f"Distance between Earth and Sun in km: {earth_sun_dist_km}")

Distance between Earth and Sun in km: 150600429.83425352


In [30]:
# ... pretty unhandy. Let's compute it in Astronomical Units (AU). It should be 1, right???
earth_sun_dist_au = spiceypy.convrt(earth_sun_dist_km, 'km', 'AU')
print(f"Distance between Earth and Sun in AU: {earth_sun_dist_au}")

Distance between Earth and Sun in AU: 1.0067016944589644


# Earth's Orbital Elements
Depending on when you compute the state vector of our home planet your will get a distance between Earth and Sun that is larger or smaller 1! That's because the Earth is not on a "100 %" perfect circle. It is moving in an elliptical orbit. The "elliptic-ness" is defined by the eccentricity e; one of the 6 Keplerian Elements that define the shape and orientation of an orbit (a 7th value determines the position of an object).

Circular Orbit: e Elliptic Orbit: 0 < e < 1 Parabolic Orbit: e = 1 Hyperbolic Orbit: e > 1



In [44]:
# First we need to set the gravitational parameter
_, grav_mu = spiceypy.bodvrd("SUN", "GM", 1)
grav_mu = grav_mu[0]
print(grav_mu)

132712440041.27939


In [45]:
# Computing the keplerian elements
(earth_peri_km,
 earth_ecc,
 earth_incl_rad,
 earth_lnode_asc_rad,
 earth_argp_rad,
 m0_rad,
 t0,
 mu) = spiceypy.oscelt(earth_state_wrt_sun, et_now, grav_mu)

print(f"Earth's eccentricity: {earth_ecc}")

Earth's eccentricity: 0.016329294484320557


In [46]:
start_date = "2025-01-01"
end_date = "2026-01-01"

comp_days = np.arange(start_date, end_date, dtype='datetime64[D]')
comp_days = comp_days.astype(str)

print(comp_days)

['2025-01-01' '2025-01-02' '2025-01-03' '2025-01-04' '2025-01-05'
 '2025-01-06' '2025-01-07' '2025-01-08' '2025-01-09' '2025-01-10'
 '2025-01-11' '2025-01-12' '2025-01-13' '2025-01-14' '2025-01-15'
 '2025-01-16' '2025-01-17' '2025-01-18' '2025-01-19' '2025-01-20'
 '2025-01-21' '2025-01-22' '2025-01-23' '2025-01-24' '2025-01-25'
 '2025-01-26' '2025-01-27' '2025-01-28' '2025-01-29' '2025-01-30'
 '2025-01-31' '2025-02-01' '2025-02-02' '2025-02-03' '2025-02-04'
 '2025-02-05' '2025-02-06' '2025-02-07' '2025-02-08' '2025-02-09'
 '2025-02-10' '2025-02-11' '2025-02-12' '2025-02-13' '2025-02-14'
 '2025-02-15' '2025-02-16' '2025-02-17' '2025-02-18' '2025-02-19'
 '2025-02-20' '2025-02-21' '2025-02-22' '2025-02-23' '2025-02-24'
 '2025-02-25' '2025-02-26' '2025-02-27' '2025-02-28' '2025-03-01'
 '2025-03-02' '2025-03-03' '2025-03-04' '2025-03-05' '2025-03-06'
 '2025-03-07' '2025-03-08' '2025-03-09' '2025-03-10' '2025-03-11'
 '2025-03-12' '2025-03-13' '2025-03-14' '2025-03-15' '2025-03-16'
 '2025-03-

In [47]:
elements_df = pd.DataFrame(comp_days, columns=["date"])

In [48]:
elements_df

,date
0,2025-01-01
1,2025-01-02
2,2025-01-03
3,2025-01-04
4,2025-01-05
...,...
360,2025-12-27
361,2025-12-28
362,2025-12-29
363,2025-12-30


In [49]:
elements_df.loc[:, "et"] = elements_df["date"].apply(lambda x: spiceypy.utc2et(x))
elements_df

,date,et
0,2025-01-01,7.889617e+08
1,2025-01-02,7.890481e+08
2,2025-01-03,7.891345e+08
3,2025-01-04,7.892209e+08
4,2025-01-05,7.893073e+08
...,...,...
360,2025-12-27,8.200657e+08
361,2025-12-28,8.201521e+08
362,2025-12-29,8.202385e+08
363,2025-12-30,8.203249e+08


In [55]:
elements_df.loc[:,"earth_state"] = elements_df["et"].apply(lambda x: spiceypy.spiceypy.spkgeo(targ = 399,
                                                                                              et = x,
                                                                                              ref="ECLIPJ2000",
                                                                                              obs=10)[0])

In [56]:
elements_df

,date,et,earth_state
0,2025-01-01,7.889617e+08,"[-26732723.48115522, 144658184.58113536, -7643..."
1,2025-01-02,7.890481e+08,"[-29302156.60364332, 144157808.32774538, -7651..."
2,2025-01-03,7.891345e+08,"[-31862317.338707764, 143612368.51572078, -767..."
3,2025-01-04,7.892209e+08,"[-34412362.275784515, 143022073.34945244, -770..."
4,2025-01-05,7.893073e+08,"[-36951461.83740085, 142387156.97229698, -7752..."
...,...,...,...
360,2025-12-27,8.200657e+08,"[-13118332.353613049, 146542083.79277813, -878..."
361,2025-12-28,8.201521e+08,"[-15721374.13557042, 146279143.5524698, -8839...."
362,2025-12-29,8.202385e+08,"[-18319270.021385916, 145970736.13510263, -888..."
363,2025-12-30,8.203249e+08,"[-20911222.427413493, 145617014.66198355, -891..."


In [58]:
col_names = ["earth_peri_km", 
             "earth_ecc", 
             "earth_incl_rad", 
             "earth_lnode_asc_rad", 
             "earth_argp_rad",
             "m0_rad", 
             "t0", 
             "mu"]

elements_df[col_names] = [spiceypy.oscelt(state, et, grav_mu) for state, et in zip(elements_df["earth_state"], elements_df["et"])]

In [59]:
x = np.degrees(elements_df["earth_incl_rad"])
y = elements_df["earth_ecc"]

N = len(x)

path_data = np.column_stack((x,y))

In [60]:
# 2. Define the Frames
# We create a list of frames. Each frame represents one "step" of the dot.
# We set traces=[1] so we only update the Dot (Trace 1), not the Line (Trace 0).
frames = [
    go.Frame(
        data=[go.Scatter(x=[path_data[k, 0]], y=[path_data[k, 1]])],
        name=str(k),
        traces=[1]
    )
    for k in range(N)
]

# 3. Create the Base Figure
fig = go.Figure(
    data=[
        # Trace 0: The Static Line
        go.Scatter(
            x=path_data[:, 0],
            y=path_data[:, 1],
            mode='lines',
            line=dict(color='gray', width=2),
            name='Path'
        ),
        # Trace 1: The Moving Dot (Initial Position)
        go.Scatter(
            x=[path_data[0, 0]],
            y=[path_data[0, 1]],
            mode='markers',
            marker=dict(color='red', size=12),
            name='Current Position'
        )
    ],
    frames=frames
)

# 4. Configure the Slider
# The slider steps tell Plotly which frame to animate to when dragged.
sliders = [dict(
    steps=[
        dict(
            method='animate',
            # The args here ensure the transition is instant (duration=0)
            # so the slider feels responsive like a UI control, not a slow movie.
            args=[[str(k)], dict(mode='immediate', frame=dict(duration=0, redraw=False), transition=dict(duration=0))],
            label=str(k)
        ) for k in range(N)
    ],
    active=0,
    transition=dict(duration=0),
    x=0, # Slider position settings
    y=0,
    currentvalue=dict(font=dict(size=12), prefix='Index: ', visible=True),
    len=1.0 # Slider length (1.0 = 100% width)
)]

# 5. Final Layout
fig.update_layout(
    title="Plotly-Only Slider Animation",
    sliders=sliders,
    xaxis_title="Incl. in deg",
    yaxis_title="Eccentricity",
    height=600,
    template="plotly_white"
)

fig.show()

## Earth’s Orbital Evolution Over Time

This graph shows the evolution of Earth’s orbit around the Sun over the period from 2025-01-01 to 2026-01-01. The x-axis represents the orbital inclination in degrees, while the y-axis represents the orbital eccentricity.

In other words, each point on the curve corresponds to a specific date, and the path traces how Earth’s orbital geometry changes over time as it moves through the year. The eccentricity measures how stretched the orbit is relative to a perfect circle (0 = circular, higher values = more elongated), while the inclination measures the tilt of Earth’s orbital plane relative to a reference plane.

The animated slider lets you step through the year day by day, showing the “current” Earth position as a red marker moving along the orbital path. The line itself represents the continuous variation of Earth’s inclination and eccentricity over the full year. For Earth, both values remain very small, which is consistent with a near-circular orbit with only a slight tilt relative to the reference frame.

This type of plot is useful for visualizing how orbital elements vary over time and for understanding the subtle changes in Earth’s heliocentric orbit caused by the dynamics of the solar system.